# SISSO 3.0.2 · Generalização entre eletrodos

Executamos o **[SISSO 3.0.2 de Runhai Ouyang](https://github.com/rouyang2017/SISSO/tree/50159de1c9388dd4e5b0bae2421df402bcad8edb)** em modo de classificação. O recorte contém as mesmas 49 aplicações e seis atributos do notebook «Modelos básicos». Em cada rodada LOEO, retiramos um eletrodo do treino, executamos o SISSO no restante e usamos o melhor descritor de dimensão 2 encontrado pelo programa.

O SISSO faz a seleção do descritor por sobreposição de domínios convexos. Para converter as coordenadas em rótulos nas aplicações deixadas de fora, ajustamos um SVM linear nos descritores padronizados do treino, com C=1 fixado para todas as divisões. O repositório oficial inclui um exemplo de SVM em utilities/SVC.py. A expressão selecionada, a padronização das coordenadas e a fronteira do SVM são recalculadas para cada eletrodo.

Referência do método: [Ouyang et al., Physical Review Materials 2, 083802 (2018)](https://doi.org/10.1103/PhysRevMaterials.2.083802).

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display
from sklearn.model_selection import LeaveOneGroupOut

PASTA = Path.cwd()
if (PASTA / 'TCC_EIS_Modelos_Basicos').is_dir():
    PASTA = PASTA / 'TCC_EIS_Modelos_Basicos'
assert (PASTA / 'dados').is_dir(), 'Abra o notebook na pasta do pacote ou do projeto.'
sys.path.insert(0, str(PASTA))
from dados_dispositivos import carregar_recorte
from sisso_oficial import localizar_sisso

df, X = carregar_recorte(PASTA / 'dados' / 'Medicoes_EIS_Por_Dispositivo_ATUALIZADO_E02.zip')
y = df.classe.to_numpy()
grupos = df.electrode.to_numpy()
EXE = localizar_sisso(PASTA.parent)

## Configuração do ajuste

Usamos os operadores soma, subtração e multiplicação, uma etapa de construção de atributos e até 30 candidatos por etapa de triagem. O ajuste é de classificação (ptype=2), com busca L0, dois grupos convexos e tolerância de fronteira de 0,001. Essas escolhas foram fixadas antes da avaliação. Os arquivos de treino são escritos em uma pasta temporária com as duas classes em blocos, conforme o formato de entrada do SISSO. Cada reconstrução do descritor é conferida contra as coordenadas de treino exportadas pelo executável.

In [2]:
from sisso_oficial import CLASSES, prever_sisso

## Avaliação LOEO

Exportamos uma previsão por aplicação para dados/previsoes_sisso_loeo.csv. O notebook de comparação usa esse arquivo para incluir **SISSO 3.0.2 + SVM linear** na figura. No ambiente Windows deste projeto, a compilação local com GNU Fortran e Open MPI usa o código oficial com adaptações para o GNU Fortran e Open MPI nos vínculos MPI e na criação dos diretórios de saída. O notebook prepara esses diretórios antes de rodar o binário.

In [3]:
previsoes = df[['application', 'electrode', 'classe']].copy()
previsoes['previsao'] = pd.Series(pd.NA, index=previsoes.index, dtype='string')
ajustes = []
for treino, teste in LeaveOneGroupOut().split(X, y, grupos):
    assert set(grupos[treino]).isdisjoint(grupos[teste])
    pred, expressoes = prever_sisso(X[treino], y[treino], X[teste], EXE)
    previsoes.loc[teste, 'previsao'] = pred
    ajustes.append((grupos[teste[0]], *expressoes, len(treino), len(teste), int((pred == y[teste]).sum())))

assert previsoes.previsao.isin(CLASSES).all() and previsoes.application.is_unique
resultado = pd.DataFrame(ajustes, columns=['Eletrodo', 'Descritor 1', 'Descritor 2', 'Treino', 'Total', 'Acertos'])
display(resultado)
acertos = int(previsoes.classe.eq(previsoes.previsao).sum())
print(f'SISSO 3.0.2 + SVM linear: {acertos}/{len(previsoes)} acertos ({acertos / len(previsoes):.1%}).')
previsoes.to_csv(PASTA / 'dados' / 'previsoes_sisso_loeo.csv', index=False)

,Eletrodo,Descritor 1,Descritor 2,Treino,Total,Acertos
0,E01,(z1-z5),(z2-z4),45,4,2
1,E03,(z2+z6),(z2*z4),45,4,2
2,E04,(z4-z6),(z2),45,4,2
3,E05,(z3+z6),(z1*z5),45,4,2
4,E07,(z2-z3),(z3-z4),31,18,10
5,E09,(z4+z6),(z2*z6),46,3,1
6,E10,(z4),(z2*z6),45,4,2
7,E13,(z2-z6),(z1-z3),45,4,2
8,E14,(z4-z6),(z2),45,4,2


SISSO 3.0.2 + SVM linear: 25/49 acertos (51.0%).
